# Eredivisie Cross Analysis 2024–25

Loads Opta F24 JSON event files from Google Drive and computes per-cross metrics:

| Metric | Description |
|---|---|
| `distance_m` | Delivery distance in metres (Opta qualifier 212) |
| `angle_deg` | Cross vector angle from pitch horizontal |
| `curve_score` | Inswing (+) / outswing (−) scaled by ball-flight qualifier |
| `speed_ms` | Proxy ball speed = distance / timestamp gap |
| `result` | `goal` \| `shot_ot` \| `shot` \| `completed` \| `cleared` \| `incomplete` |
| `xt_start/end/delta` | Expected Threat (Karun Singh 12×8 grid) |
| `goal_diff` | Crossing team minus opposition at time of cross |
| `vaep_value` | Atomic VAEP (ΔP_score − ΔP_concede) |
| `rapm` | Regularised Adjusted Plus-Minus via ridge regression |

**Drive path expected:** `My Drive/Event data/Eredivisie 2024-2025/`
Filename format: `YYYY-MM-DD_HomeTeam - AwayTeam.json`

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install mplsoccer pandas numpy matplotlib --quiet

In [ ]:
# ── 2. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 3. Imports ────────────────────────────────────────────────────────────────
import json
import math
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from mplsoccer import Pitch, VerticalPitch

print(f'pandas {pd.__version__} | numpy {np.__version__} | matplotlib {matplotlib.__version__}')

In [ ]:
# ── 4. Configuration ──────────────────────────────────────────────────────────

# Path to Eredivisie JSON files on Drive
DATA_DIR = '/content/drive/My Drive/Event data/Eredivisie 2024-2025'

# Filter to a specific player — set to None to include all players
# Examples: 'B. Brobbey', 'L. de Jong', 'S. Gimenez'
PLAYER_FILTER = None

# Filter to a specific team — set to None for all teams
# Example: 'Ajax', 'PSV', 'Feyenoord'
TEAM_FILTER = None

# Minimum crosses to include a player in ranked charts
MIN_CROSSES = 5

# Colour palette
NAVY   = '#0D1B2A'
TEAL   = '#1B7A78'
GOLD   = '#E8C33A'
GREEN  = '#1DB954'
RED    = '#E05252'
WHITE  = '#FFFFFF'

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': NAVY,
    'axes.facecolor':   '#111E2B',
    'axes.edgecolor':   '#2A3E52',
    'text.color':       '#EEEEEE',
    'axes.labelcolor':  '#CCCCCC',
    'xtick.color':      '#AAAAAA',
    'ytick.color':      '#AAAAAA',
    'grid.color':       '#1E2E3E',
    'grid.alpha':       0.6,
    'font.family':      'DejaVu Sans',
})

print('Config ready.')
print(f'Data directory : {DATA_DIR}')
print(f'Player filter  : {PLAYER_FILTER or "all"}')
print(f'Team filter    : {TEAM_FILTER or "all"}')

---
## Section 1 — Constants & Helper Functions

In [ ]:
# ── Pitch constants ───────────────────────────────────────────────────────────
PITCH_LEN_M = 105.0
PITCH_WID_M = 68.0

# ── xT grid: Karun Singh 12 cols × 8 rows ─────────────────────────────────────
# Rows = y-bands (0→100), Cols = x-bands (own goal → attacking goal)
XT_GRID = np.array([
    [0.00638, 0.00349, 0.00175, 0.00172, 0.00172, 0.00289, 0.00447, 0.00584, 0.00757, 0.01102, 0.01827, 0.02956],
    [0.00826, 0.00506, 0.00333, 0.00309, 0.00304, 0.00461, 0.00577, 0.00725, 0.01099, 0.01842, 0.03045, 0.04692],
    [0.01223, 0.00735, 0.00455, 0.00405, 0.00404, 0.00563, 0.00713, 0.00947, 0.01498, 0.02471, 0.04262, 0.07512],
    [0.01657, 0.00931, 0.00576, 0.00480, 0.00484, 0.00621, 0.00830, 0.01174, 0.01826, 0.03121, 0.05765, 0.10510],
    [0.01657, 0.00931, 0.00576, 0.00480, 0.00484, 0.00621, 0.00830, 0.01174, 0.01826, 0.03121, 0.05765, 0.10510],
    [0.01223, 0.00735, 0.00455, 0.00405, 0.00404, 0.00563, 0.00713, 0.00947, 0.01498, 0.02471, 0.04262, 0.07512],
    [0.00826, 0.00506, 0.00333, 0.00309, 0.00304, 0.00461, 0.00577, 0.00725, 0.01099, 0.01842, 0.03045, 0.04692],
    [0.00638, 0.00349, 0.00175, 0.00172, 0.00172, 0.00289, 0.00447, 0.00584, 0.00757, 0.01102, 0.01827, 0.02956],
])


def q_val(event, qid):
    """Return qualifier value by ID, or None."""
    for q in event.get('qualifier', []):
        if q['qualifierId'] == qid:
            return q.get('value')
    return None

def has_q(event, qid):
    return any(q['qualifierId'] == qid for q in event.get('qualifier', []))

def to_f(v, default=None):
    try:
        return float(v)
    except (TypeError, ValueError):
        return default


def normalize_coords(x, y, ex, ey, is_home, period_id):
    """
    Flip coordinates so the crossing team always attacks toward x=100.
    Home team in period 1: already attacking x→100, no flip.
    Home team in period 2: flip needed.
    Away team: opposite.
    """
    flip = (is_home and period_id == 2) or (not is_home and period_id == 1)
    if flip:
        x = 100.0 - x
        y = 100.0 - y
        if ex is not None: ex = 100.0 - ex
        if ey is not None: ey = 100.0 - ey
    return x, y, ex, ey


print('Constants and helpers loaded.')

---
## Section 2 — Metric Functions

In [ ]:
# ── Distance & angle ─────────────────────────────────────────────────────────

def dist_m(x1, y1, x2, y2):
    dx = (x2 - x1) / 100.0 * PITCH_LEN_M
    dy = (y2 - y1) / 100.0 * PITCH_WID_M
    return math.hypot(dx, dy)


def cross_angle(x1, y1, x2, y2):
    """Angle of cross vector in degrees. 0° = straight toward goal."""
    dx = (x2 - x1) / 100.0 * PITCH_LEN_M
    dy = (y2 - y1) / 100.0 * PITCH_WID_M
    return math.degrees(math.atan2(dy, dx))


# ── Curve / swing ─────────────────────────────────────────────────────────────

def curve_score(x, y, ex, ey, flight_metric):
    """
    Estimate inswing (+) vs outswing (−) using side of pitch.
    Crosses from the right flank (y < 50) tend to be outswing for a
    right-footed passer; from the left tend to be inswing.
    Result is scaled by Opta qualifier 213 (ball-flight metric, typically 1–7).
    Without tracking data this is an approximation.
    """
    if ex is None or ey is None:
        return 0.0
    from_right = y < 50.0
    direction  = -1.0 if from_right else 1.0  # right-flank default = outswing
    scale      = min((flight_metric or 1.0) / 5.0, 1.0)
    return round(direction * scale, 4)


# ── Speed proxy ───────────────────────────────────────────────────────────────

def speed_proxy(event, next_ev):
    """m/s estimate using Opta Q212 distance and event timestamp gap."""
    if next_ev is None:
        return None
    d = to_f(q_val(event, 212))
    if not d or d <= 0:
        return None
    t1 = event.get('timeMin', 0) * 60 + event.get('timeSec', 0)
    t2 = next_ev.get('timeMin', 0) * 60 + next_ev.get('timeSec', 0)
    dt = t2 - t1
    if dt <= 0 or dt > 8:
        return None
    return round(d / dt, 2)


# ── Expected Threat ───────────────────────────────────────────────────────────

def get_xt(x, y):
    col = min(int(x / 100.0 * 12), 11)
    row = min(int(y / 100.0 *  8),  7)
    return float(XT_GRID[row, col])


# ── Cross result (5-event lookahead) ──────────────────────────────────────────

def cross_result(events, idx):
    ev   = events[idx]
    team = ev.get('contestantId')
    if ev.get('outcome', 0) == 0:
        return 'incomplete'
    for fev in events[idx + 1 : idx + 6]:
        t = fev.get('typeId')
        if fev.get('contestantId') != team:
            return 'cleared'
        if t == 16:       return 'goal'
        if t == 15:       return 'shot_ot'
        if t in {13, 14}: return 'shot'
    return 'completed'


# ── Goal difference at time of cross ─────────────────────────────────────────

def goal_diff_at(events, idx, team_id):
    a, o = 0, 0
    for ev in events[:idx]:
        if ev.get('typeId') == 16:
            if ev.get('contestantId') == team_id: a += 1
            else:                                  o += 1
    return a - o


# ── Atomic VAEP ───────────────────────────────────────────────────────────────

def _score_prob(x, y):
    """
    P(team scores in next ~5 actions) from position.
    Simplified logistic model calibrated to empirical rates.
    A full VAEP model uses gradient boosting on historical sequences.
    """
    d = math.hypot((1.0 - x / 100) * PITCH_LEN_M,
                   (0.5 - y / 100) * PITCH_WID_M)
    central = 1.0 - 2.0 * abs(y / 100 - 0.5)
    p = 0.18 * math.exp(-0.07 * max(d - 5, 0))
    return min(p * (0.6 + 0.4 * central), 1.0)


def atomic_vaep(events, idx):
    ev       = events[idx]
    prev_ev  = events[idx - 1] if idx > 0 else ev
    team     = ev.get('contestantId')

    x0, y0 = prev_ev.get('x', 50), prev_ev.get('y', 50)
    x1, y1 = ev.get('x', 50),      ev.get('y', 50)
    ex = to_f(q_val(ev, 140)) or x1
    ey = to_f(q_val(ev, 141)) or y1

    p_sc_pre  = _score_prob(x0, y0)
    p_sc_post = _score_prob(ex, ey)
    p_co_pre  = _score_prob(100 - x0, y0)
    p_co_post = _score_prob(100 - ex,  ey)

    if ev.get('outcome', 0) == 0:   # incomplete cross → ball lost
        p_sc_post *= 0.15
        p_co_post  = min(p_co_post * 1.4, 1.0)

    v_sc = p_sc_post - p_sc_pre
    v_co = p_co_post - p_co_pre

    scored = conceded = False
    for fev in events[idx + 1 : idx + 6]:
        if fev.get('typeId') == 16:
            if fev.get('contestantId') == team: scored   = True
            else:                               conceded = True
            break

    return {
        'vaep_value':       round(v_sc - v_co, 5),
        'p_score_delta':    round(v_sc, 5),
        'p_concede_delta':  round(v_co, 5),
        'scored_in_seq':    scored,
        'conceded_in_seq':  conceded,
    }


print('Metric functions loaded.')

---
## Section 3 — RAPM (Regularised Adjusted Plus-Minus)

In [ ]:
# Plus-minus score per result outcome
RESULT_PM = {
    'goal':       1.00,
    'shot_ot':    0.60,
    'shot':       0.35,
    'completed':  0.15,
    'cleared':   -0.10,
    'incomplete': -0.20,
}


def compute_rapm(df):
    """
    Ridge regression (λ=2) of cross plus-minus on player-season dummies.
    Solves: β = (XᵀX + λI)⁻¹ Xᵀy

    Each unique (player, season) pair gets a RAPM coefficient representing
    their regularised cross contribution per delivery.
    """
    df = df.copy()
    df['pm_score'] = df['result'].map(RESULT_PM).fillna(0.0)
    df['ps_key']   = df['player'] + ' | ' + df['season']

    keys = sorted(df['ps_key'].unique())
    if len(keys) == 1:
        # Single unit: Bayesian shrinkage
        n = len(df)
        df['rapm'] = round(df['pm_score'].mean() * n / (n + 10), 5)
        return df

    key_idx = {k: i for i, k in enumerate(keys)}
    X = np.zeros((len(df), len(keys)))
    for row_i, k in enumerate(df['ps_key']):
        X[row_i, key_idx[k]] = 1.0
    y = df['pm_score'].values

    lam  = 2.0
    beta = np.linalg.solve(X.T @ X + lam * np.eye(len(keys)), X.T @ y)
    rapm_map = {k: round(float(beta[i]), 5) for k, i in key_idx.items()}
    df['rapm'] = df['ps_key'].map(rapm_map)
    df.drop(columns=['ps_key'], inplace=True)
    return df


print('RAPM function loaded.')

---
## Section 4 — Load JSON Files & Extract Crosses

In [ ]:
def load_crosses(data_dir, player_filter=None, team_filter=None):
    """
    Walk all JSON files in data_dir, identify crosses (Opta qualifier 2),
    normalise coordinates, and compute all metrics.
    """
    json_files = sorted(glob.glob(os.path.join(data_dir, '*.json')))
    if not json_files:
        raise FileNotFoundError(
            f'No JSON files found in:\n  {data_dir}\n'
            f'Check your Drive path and ensure the folder is accessible.'
        )
    print(f'Found {len(json_files)} JSON files in {data_dir}')

    rows = []

    for filepath in json_files:
        fname = os.path.basename(filepath)

        try:
            with open(filepath) as f:
                data = json.load(f)
        except Exception as e:
            print(f'  [SKIP] {fname}: {e}')
            continue

        events = data.get('event', [])
        if not events:
            continue

        # Parse home / away teams from filename
        parts       = fname.replace('.json', '').split('_', 1)
        date_str    = parts[0] if parts else 'Unknown'
        teams_str   = parts[1] if len(parts) > 1 else ''
        home_team, _, away_team = teams_str.partition(' - ')
        home_team   = home_team.strip()
        away_team   = away_team.strip()

        if team_filter and team_filter.lower() not in (
            home_team.lower(), away_team.lower()
        ):
            continue

        # Build team_id → (team_name, is_home) lookup
        team_lookup = {}   # contestantId → (team_name, is_home)
        for ev in events:
            tid = ev.get('contestantId')
            if tid and tid not in team_lookup:
                pname = ev.get('playerName', '')
                # Match player event to team name via side inference later;
                # we populate team names from the filename order instead.
                team_lookup[tid] = None   # placeholder

        # We can't reliably map contestantId → home/away from F24 alone
        # without a separate metadata file, so we use a two-pass approach:
        # count goals per contestantId and match to matchDetails scores.
        home_goals_ft = data.get('matchDetails', {}).get('scores', {}).get('ft', {}).get('home', 0)
        away_goals_ft = data.get('matchDetails', {}).get('scores', {}).get('ft', {}).get('away', 0)

        goal_events  = [ev for ev in events if ev.get('typeId') == 16]
        team_goals   = {}
        for ev in goal_events:
            tid = ev.get('contestantId')
            team_goals[tid] = team_goals.get(tid, 0) + 1

        # Assign home/away by matching goal counts
        team_ids  = list(team_lookup.keys())
        home_id   = None
        away_id   = None

        if len(team_ids) == 2:
            t0, t1 = team_ids[0], team_ids[1]
            g0 = team_goals.get(t0, 0)
            g1 = team_goals.get(t1, 0)
            # Best guess: team with home_goals_ft goals = home
            if g0 == home_goals_ft:
                home_id, away_id = t0, t1
            elif g1 == home_goals_ft:
                home_id, away_id = t1, t0
            else:
                # Fallback: first team found = home
                home_id, away_id = t0, t1
        elif len(team_ids) >= 1:
            home_id = team_ids[0]
            away_id = team_ids[1] if len(team_ids) > 1 else team_ids[0]

        team_id_to_name = {}
        if home_id: team_id_to_name[home_id] = home_team
        if away_id: team_id_to_name[away_id] = away_team

        # ── Iterate events ────────────────────────────────────────────────────
        for idx, ev in enumerate(events):
            if ev.get('typeId') != 1:
                continue
            if not has_q(ev, 2):          # qualifier 2 = cross
                continue

            pname = ev.get('playerName', 'Unknown')
            if player_filter and player_filter.lower() not in pname.lower():
                continue

            team_id  = ev.get('contestantId')
            is_home  = (team_id == home_id)
            team_name = team_id_to_name.get(team_id, 'Unknown')

            if team_filter and team_filter.lower() not in team_name.lower():
                continue

            raw_x  = ev.get('x', 0.0)
            raw_y  = ev.get('y', 0.0)
            raw_ex = to_f(q_val(ev, 140))
            raw_ey = to_f(q_val(ev, 141))

            nx, ny, nex, ney = normalize_coords(
                raw_x, raw_y, raw_ex, raw_ey, is_home, ev.get('periodId', 1)
            )

            dist_q       = to_f(q_val(ev, 212))
            flight_metic = to_f(q_val(ev, 213), default=1.0)
            dest_zone    = q_val(ev, 56) or 'Unknown'

            d_m  = dist_q if dist_q else (dist_m(nx, ny, nex, ney) if nex else None)
            ang  = cross_angle(nx, ny, nex, ney) if nex else None
            crv  = curve_score(nx, ny, nex, ney, flight_metic)
            spd  = speed_proxy(ev, events[idx + 1] if idx + 1 < len(events) else None)

            result = cross_result(events, idx)

            xt_s = get_xt(nx, ny)
            xt_e = get_xt(nex, ney) if nex is not None else None
            xt_d = round(xt_e - xt_s, 6) if xt_e is not None else None

            gd   = goal_diff_at(events, idx, team_id)
            vaep = atomic_vaep(events, idx)

            rows.append({
                'date':             date_str,
                'season':           '2024-25',
                'player':           pname,
                'team':             team_name,
                'opponent':         away_team if is_home else home_team,
                'home_away':        'H' if is_home else 'A',
                'period':           ev.get('periodId'),
                'minute':           ev.get('timeMin'),
                'second':           ev.get('timeSec'),
                'x':                round(nx, 2),
                'y':                round(ny, 2),
                'end_x':            round(nex, 2) if nex is not None else None,
                'end_y':            round(ney, 2) if ney is not None else None,
                'dest_zone':        dest_zone,
                'outcome':          ev.get('outcome', 0),
                'distance_m':       round(d_m, 2) if d_m else None,
                'angle_deg':        round(ang, 1) if ang else None,
                'curve_score':      crv,
                'flight_metric':    flight_metic,
                'speed_ms':         spd,
                'result':           result,
                'xt_start':         round(xt_s, 6),
                'xt_end':           round(xt_e, 6) if xt_e is not None else None,
                'xt_delta':         xt_d,
                'goal_diff':        gd,
                'vaep_value':       vaep['vaep_value'],
                'p_score_delta':    vaep['p_score_delta'],
                'p_concede_delta':  vaep['p_concede_delta'],
                'scored_in_seq':    vaep['scored_in_seq'],
                'conceded_in_seq':  vaep['conceded_in_seq'],
            })

    if not rows:
        raise ValueError('No crosses extracted. Check player/team filter names.')

    df = pd.DataFrame(rows)
    df = compute_rapm(df)
    return df


# ── Run ───────────────────────────────────────────────────────────────────────
df = load_crosses(DATA_DIR, player_filter=PLAYER_FILTER, team_filter=TEAM_FILTER)
print(f'\nTotal crosses extracted: {len(df):,}')
print(f'Unique players : {df["player"].nunique()}')
print(f'Unique teams   : {df["team"].nunique()}')
df.head()

---
## Section 5 — Summary Statistics

In [ ]:
sep = '=' * 62
print(f'\n{sep}')
print('  CROSS ANALYSIS — EREDIVISIE 2024-25')
print(sep)
print(f'  Total crosses          : {len(df):>6}')
print(f'  Successful (outcome=1) : {(df["outcome"]==1).sum():>6}  '
      f'({(df["outcome"]==1).mean():.1%})')

print(f'\n  Result breakdown:')
for r, cnt in df['result'].value_counts().items():
    print(f'    {r:<15} {cnt:>5}  ({cnt/len(df):.1%})')

print(f'\n  Average metrics:')
for col, label in [
    ('distance_m',  'Distance (m)'),
    ('angle_deg',   'Angle (°)'),
    ('curve_score', 'Curve score'),
    ('speed_ms',    'Speed proxy (m/s)'),
    ('xt_delta',    'xT delta'),
    ('vaep_value',  'Atomic VAEP'),
    ('rapm',        'RAPM'),
    ('goal_diff',   'Goal diff at cross'),
]:
    if df[col].notna().any():
        print(f'    {label:<24} {df[col].mean():>8.3f}')
print(sep)

In [ ]:
# ── Per-player ranking table (min MIN_CROSSES) ────────────────────────────────
player_stats = (
    df.groupby('player')
    .agg(
        team       = ('team', 'first'),
        crosses    = ('result', 'count'),
        success_rt = ('outcome', 'mean'),
        dist_avg   = ('distance_m', 'mean'),
        angle_avg  = ('angle_deg', 'mean'),
        xt_delta   = ('xt_delta', 'mean'),
        vaep_avg   = ('vaep_value', 'mean'),
        rapm       = ('rapm', 'mean'),
        goals_from = ('result', lambda x: (x == 'goal').sum()),
        shots_from = ('result', lambda x: x.isin(['goal','shot_ot','shot']).sum()),
    )
    .reset_index()
    .query(f'crosses >= {MIN_CROSSES}')
    .sort_values('vaep_avg', ascending=False)
    .round(3)
)

print(f'Players with ≥{MIN_CROSSES} crosses: {len(player_stats)}')
player_stats.head(20)

---
## Section 6 — Visualisations

In [ ]:
# ── 6a. Cross origin heatmap on pitch ────────────────────────────────────────
pitch = Pitch(pitch_type='opta', pitch_color=NAVY, line_color='#2A3E52',
              line_zorder=2)
fig, ax = pitch.draw(figsize=(12, 8))
fig.patch.set_facecolor(NAVY)

# Separate successful and incomplete for colouring
ok  = df[df['outcome'] == 1]
bad = df[df['outcome'] == 0]

pitch.scatter(bad['x'], bad['y'], ax=ax, color=RED,   alpha=0.3,
              s=20, zorder=3, label='Incomplete')
pitch.scatter(ok['x'],  ok['y'],  ax=ax, color=GREEN, alpha=0.5,
              s=25, zorder=4, label='Successful')

# Draw arrows for successful crosses that have an end point
ok_xy = ok.dropna(subset=['end_x', 'end_y'])
for _, row in ok_xy.head(200).iterrows():
    pitch.arrows(row['x'], row['y'], row['end_x'], row['end_y'],
                 ax=ax, color=TEAL, alpha=0.25, width=0.8, headwidth=3)

ax.set_title('Cross Origins — Eredivisie 2024-25', fontsize=14,
             fontweight='bold', color=GOLD, pad=12)
ax.legend(loc='upper left', fontsize=9,
          facecolor='#1A2B3C', edgecolor='#2A3E52')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6b. Delivery destination heatmap (successful only) ───────────────────────
dest = df.dropna(subset=['end_x', 'end_y'])
dest = dest[dest['outcome'] == 1]

pitch2 = Pitch(pitch_type='opta', pitch_color=NAVY, line_color='#2A3E52', line_zorder=2)
fig, ax = pitch2.draw(figsize=(12, 8))
fig.patch.set_facecolor(NAVY)

_, _, _ = pitch2.bin_statistic(
    dest['end_x'], dest['end_y'], statistic='count', bins=(18, 12))

cmap = LinearSegmentedColormap.from_list(
    'xtcmap', [NAVY, '#1B7A78', GOLD, '#E05252'])

bs = pitch2.bin_statistic(dest['end_x'], dest['end_y'],
                           statistic='count', bins=(18, 12))
hm = pitch2.heatmap(bs, ax=ax, cmap=cmap, edgecolors=NAVY, linewidth=0.3)
plt.colorbar(hm, ax=ax, shrink=0.6, label='Deliveries into zone')

ax.set_title('Cross Delivery Destinations — Successful Crosses',
             fontsize=13, fontweight='bold', color=GOLD, pad=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6c. Result breakdown bar chart ───────────────────────────────────────────
result_order  = ['goal', 'shot_ot', 'shot', 'completed', 'cleared', 'incomplete']
result_colors = [GREEN, '#5BA85A', TEAL, GOLD, '#E07A3A', RED]
result_counts = df['result'].value_counts().reindex(result_order).fillna(0)

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(NAVY)
ax.set_facecolor('#111E2B')

bars = ax.bar(result_counts.index, result_counts.values,
              color=result_colors, edgecolor='#2A3E52', linewidth=0.7, width=0.65)
for bar, val in zip(bars, result_counts.values):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{int(val)}\n({pct:.1f}%)', ha='center', va='bottom',
            fontsize=9, color=WHITE)

ax.set_title('Cross Results — Eredivisie 2024-25',
             fontsize=13, fontweight='bold', color=GOLD, pad=12)
ax.set_ylabel('Count', color='#CCCCCC')
ax.grid(axis='y', alpha=0.3, color='#2A3E52')
for sp in ['top', 'right', 'left', 'bottom']:
    ax.spines[sp].set_color('#2A3E52')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6d. Distance distribution by result ──────────────────────────────────────
dist_data = df.dropna(subset=['distance_m'])

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor(NAVY)
ax.set_facecolor('#111E2B')

for res, col in zip(['goal', 'shot_ot', 'shot', 'cleared', 'incomplete'],
                    [GREEN, '#5BA85A', TEAL, '#E07A3A', RED]):
    sub = dist_data[dist_data['result'] == res]['distance_m']
    if len(sub) > 2:
        ax.hist(sub, bins=20, alpha=0.45, color=col, label=res, density=True)

ax.set_xlabel('Cross Distance (m)', color='#CCCCCC', fontsize=11)
ax.set_ylabel('Density', color='#CCCCCC', fontsize=11)
ax.set_title('Cross Distance Distribution by Result',
             fontsize=13, fontweight='bold', color=GOLD, pad=12)
ax.legend(fontsize=9, facecolor='#1A2B3C', edgecolor='#2A3E52')
ax.grid(alpha=0.25, color='#2A3E52')
for sp in ['top', 'right']:
    ax.spines[sp].set_visible(False)
for sp in ['left', 'bottom']:
    ax.spines[sp].set_color('#2A3E52')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6e. xT delta vs VAEP scatter by result ────────────────────────────────────
scatter_df = df.dropna(subset=['xt_delta', 'vaep_value'])

result_col_map = {
    'goal':       GREEN,
    'shot_ot':    '#5BA85A',
    'shot':       TEAL,
    'completed':  GOLD,
    'cleared':    '#E07A3A',
    'incomplete': RED,
}

fig, ax = plt.subplots(figsize=(11, 7))
fig.patch.set_facecolor(NAVY)
ax.set_facecolor('#111E2B')

for res, col in result_col_map.items():
    sub = scatter_df[scatter_df['result'] == res]
    if len(sub):
        ax.scatter(sub['xt_delta'], sub['vaep_value'],
                   color=col, alpha=0.55, s=22, label=res, zorder=3)

ax.axhline(0, color='#AAAAAA', linewidth=0.8, linestyle='--', alpha=0.6)
ax.axvline(0, color='#AAAAAA', linewidth=0.8, linestyle='--', alpha=0.6)

ax.set_xlabel('xT Delta (Expected Threat Gained)', color='#CCCCCC', fontsize=11)
ax.set_ylabel('Atomic VAEP', color='#CCCCCC', fontsize=11)
ax.set_title('xT Delta vs Atomic VAEP — Coloured by Cross Result',
             fontsize=13, fontweight='bold', color=GOLD, pad=12)
ax.legend(fontsize=9, facecolor='#1A2B3C', edgecolor='#2A3E52')
ax.grid(alpha=0.25, color='#2A3E52')
for sp in ['top', 'right']:
    ax.spines[sp].set_visible(False)
for sp in ['left', 'bottom']:
    ax.spines[sp].set_color('#2A3E52')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6f. Top crossers by VAEP (horizontal bar) ─────────────────────────────────
top_crossers = (
    player_stats
    .nlargest(20, 'vaep_avg')
    .sort_values('vaep_avg', ascending=True)
)

fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor(NAVY)
ax.set_facecolor('#111E2B')

y      = range(len(top_crossers))
colors = [GREEN if v > 0 else RED for v in top_crossers['vaep_avg']]
ax.barh(y, top_crossers['vaep_avg'], color=colors, alpha=0.85,
        edgecolor='#2A3E52', linewidth=0.6, height=0.7)
ax.axvline(0, color='#AAAAAA', linewidth=0.8)

labels = [f"{row['player']} ({row['team']}, n={int(row['crosses'])})"
          for _, row in top_crossers.iterrows()]
ax.set_yticks(list(y))
ax.set_yticklabels(labels, fontsize=8, color=WHITE)
ax.set_xlabel('Mean Atomic VAEP per Cross', color='#CCCCCC', fontsize=10)
ax.set_title(f'Top 20 Crossers by Atomic VAEP — Eredivisie 2024-25 (min {MIN_CROSSES} crosses)',
             fontsize=12, fontweight='bold', color=GOLD, pad=12)
ax.grid(axis='x', alpha=0.25, color='#2A3E52')
for sp in ['top', 'right']:
    ax.spines[sp].set_visible(False)
for sp in ['left', 'bottom']:
    ax.spines[sp].set_color('#2A3E52')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6g. xT delta by goal difference context ──────────────────────────────────
gd_xt = (
    df.dropna(subset=['xt_delta'])
    .groupby('goal_diff')['xt_delta']
    .agg(['mean', 'count'])
    .reset_index()
    .query('count >= 5')
    .sort_values('goal_diff')
)

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(NAVY)
ax.set_facecolor('#111E2B')

bar_cols = [GREEN if v >= 0 else RED for v in gd_xt['goal_diff']]
ax.bar(gd_xt['goal_diff'].astype(str), gd_xt['mean'],
       color=bar_cols, alpha=0.85, edgecolor='#2A3E52', linewidth=0.7)
ax.axhline(df['xt_delta'].mean(), color=GOLD, linestyle='--',
           linewidth=1.2, label=f'Overall mean xT delta ({df["xt_delta"].mean():.4f})')

ax.set_xlabel('Goal Difference at Time of Cross (team perspective)',
              color='#CCCCCC', fontsize=10)
ax.set_ylabel('Mean xT Delta', color='#CCCCCC', fontsize=10)
ax.set_title('Cross xT Delta by Game State',
             fontsize=13, fontweight='bold', color=GOLD, pad=12)
ax.legend(fontsize=9, facecolor='#1A2B3C', edgecolor='#2A3E52')
ax.grid(axis='y', alpha=0.25, color='#2A3E52')
for sp in ['top', 'right']:
    ax.spines[sp].set_visible(False)
for sp in ['left', 'bottom']:
    ax.spines[sp].set_color('#2A3E52')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6h. Speed proxy vs distance scatter ──────────────────────────────────────
spd_df = df.dropna(subset=['speed_ms', 'distance_m'])

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(NAVY)
ax.set_facecolor('#111E2B')

sc = ax.scatter(
    spd_df['distance_m'], spd_df['speed_ms'],
    c      = spd_df['xt_delta'],
    cmap   = 'RdYlGn',
    alpha  = 0.55,
    s      = 22,
    zorder = 3,
)
cbar = plt.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label('xT Delta', color='#CCCCCC', fontsize=9)
cbar.ax.yaxis.set_tick_params(color='#AAAAAA')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#AAAAAA')

ax.set_xlabel('Cross Distance (m)', color='#CCCCCC', fontsize=11)
ax.set_ylabel('Speed Proxy (m/s)', color='#CCCCCC', fontsize=11)
ax.set_title('Cross Speed vs Distance — Coloured by xT Delta',
             fontsize=13, fontweight='bold', color=GOLD, pad=12)
ax.grid(alpha=0.25, color='#2A3E52')
for sp in ['top', 'right']:
    ax.spines[sp].set_visible(False)
for sp in ['left', 'bottom']:
    ax.spines[sp].set_color('#2A3E52')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6i. RAPM vs xT delta per player (bubble = cross volume) ──────────────────
fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor(NAVY)
ax.set_facecolor('#111E2B')

sc = ax.scatter(
    player_stats['xt_delta'],
    player_stats['rapm'],
    s      = np.clip(player_stats['crosses'] * 3, 30, 300),
    c      = player_stats['vaep_avg'],
    cmap   = 'RdYlGn',
    alpha  = 0.75,
    zorder = 3,
    edgecolors = 'none',
)
cbar = plt.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label('Avg Atomic VAEP', color='#CCCCCC', fontsize=9)
cbar.ax.yaxis.set_tick_params(color='#AAAAAA')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#AAAAAA')

ax.axhline(0, color='#AAAAAA', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axvline(player_stats['xt_delta'].mean(), color=GOLD, linewidth=0.8,
           linestyle='--', alpha=0.5, label='Mean xT delta')

# Annotate top 10 by vaep_avg
for _, row in player_stats.nlargest(10, 'vaep_avg').iterrows():
    ax.annotate(
        row['player'],
        xy     = (row['xt_delta'], row['rapm']),
        xytext = (5, 3),
        textcoords = 'offset points',
        fontsize   = 7.5,
        color      = WHITE,
        alpha      = 0.9,
    )

ax.set_xlabel('Mean xT Delta per Cross', color='#CCCCCC', fontsize=11)
ax.set_ylabel('RAPM per Cross', color='#CCCCCC', fontsize=11)
ax.set_title(f'RAPM vs xT Delta — Eredivisie Crossers (min {MIN_CROSSES} crosses)\n'
             f'Bubble size = cross volume; colour = Atomic VAEP',
             fontsize=12, fontweight='bold', color=GOLD, pad=12)
ax.legend(fontsize=8, facecolor='#1A2B3C', edgecolor='#2A3E52')
ax.grid(alpha=0.25, color='#2A3E52')
for sp in ['top', 'right']:
    ax.spines[sp].set_visible(False)
for sp in ['left', 'bottom']:
    ax.spines[sp].set_color('#2A3E52')
plt.tight_layout()
plt.show()

---
## Section 7 — Export Results

In [ ]:
# ── Save CSVs to Drive ────────────────────────────────────────────────────────
output_dir = os.path.join('/content/drive/My Drive', 'Event data', 'Eredivisie 2024-2025')
os.makedirs(output_dir, exist_ok=True)

raw_out    = os.path.join(output_dir, 'cross_analysis_all.csv')
player_out = os.path.join(output_dir, 'cross_analysis_by_player.csv')

df.to_csv(raw_out, index=False)
player_stats.to_csv(player_out, index=False)

print(f'Raw cross data  → {raw_out}')
print(f'Player rankings → {player_out}')
print(f'\nColumns in raw file:')
print(list(df.columns))

In [ ]:
# ── Optional: also download to local Colab session ────────────────────────────
from google.colab import files

local_csv = '/content/cross_analysis_eredivisie.csv'
df.to_csv(local_csv, index=False)
files.download(local_csv)
print('Download started.')

---
## Section 8 — Expected Cross Model (xC)

Two-stage logistic regression:
- **Stage 1** `P(completed)` — origin features → will this cross reach a teammate?
- **Stage 2** `P(dangerous | completed)` — destination features → will it generate a shot?
- **xC = Stage 1 × Stage 2**

AUC ~0.63–0.68 is expected without tracking data (no box occupancy or defensive shape).

In [ ]:
# ── 8a. Install scikit-learn ──────────────────────────────────────────────────
!pip install scikit-learn --quiet
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve
from sklearn.pipeline import Pipeline
from matplotlib.gridspec import GridSpec
print('scikit-learn ready.')

In [ ]:
# ── 8b. Feature engineering ───────────────────────────────────────────────────

def _dist_to_goal(x_opta, y_opta):
    dx = (1.0 - x_opta / 100) * PITCH_LEN_M
    dy = (0.5 - y_opta / 100) * PITCH_WID_M
    return math.hypot(dx, dy)


def build_stage1_features(df):
    """Origin-only features — available before the ball leaves the foot."""
    f = pd.DataFrame(index=df.index)
    f['x_norm']          = df['x'] / 100.0
    f['y_norm']          = df['y'] / 100.0
    f['distance_m']      = df['distance_m'].fillna(df['distance_m'].median())
    f['flight_metric']   = df['flight_metric'].fillna(1.0)
    f['curve_score']     = df['curve_score']
    f['goal_diff']       = df['goal_diff'].clip(-3, 3)
    f['minute']          = df['minute'] / 90.0
    f['dist_to_goal_m']  = df.apply(lambda r: _dist_to_goal(r['x'], r['y']), axis=1)
    f['is_deep']         = (df['x'] >= 85).astype(float)
    f['from_right']      = (df['y'] < 50).astype(float)
    f['centrality_orig'] = 1.0 - 2.0 * (df['y'] / 100.0 - 0.5).abs()
    return f


def build_stage2_features(df):
    """Destination features — where did the cross land?"""
    f = pd.DataFrame(index=df.index)
    f['end_x_norm']        = df['end_x'] / 100.0
    f['end_y_norm']        = df['end_y'] / 100.0
    f['centrality_dest']   = 1.0 - 2.0 * (df['end_y'] / 100.0 - 0.5).abs()
    f['dist_to_goal_dest'] = df.apply(lambda r: _dist_to_goal(r['end_x'], r['end_y']), axis=1)
    near_post_y, far_post_y = 31.3, 68.7
    f['near_post_dist'] = df.apply(
        lambda r: math.hypot((1.0 - r['end_x']/100)*PITCH_LEN_M,
                             (near_post_y/100 - r['end_y']/100)*PITCH_WID_M), axis=1)
    f['far_post_dist']  = df.apply(
        lambda r: math.hypot((1.0 - r['end_x']/100)*PITCH_LEN_M,
                             (far_post_y/100  - r['end_y']/100)*PITCH_WID_M), axis=1)
    f['into_box']       = ((df['end_x'] >= 83) & (df['end_y'] >= 21) & (df['end_y'] <= 79)).astype(float)
    f['distance_m']     = df['distance_m'].fillna(df['distance_m'].median())
    f['angle_deg']      = df['angle_deg'].fillna(0.0)
    f['flight_metric']  = df['flight_metric'].fillna(1.0)
    f['dest_central']   = (df['dest_zone'] == 'Center').astype(float)
    return f


def make_pipe(C=1.0, class_weight=None):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(C=C, class_weight=class_weight,
                                     max_iter=1000, solver='lbfgs', random_state=42)),
    ])

print('Feature functions defined.')

In [ ]:
# ── 8c. Train Stage 1 & Stage 2 ───────────────────────────────────────────────
DANGEROUS = ['goal', 'shot_ot', 'shot']

X1 = build_stage1_features(df).values
y1 = df['outcome'].values

# Stage 2 uses only completed crosses
comp = df[df['outcome'] == 1].copy()
X2   = build_stage2_features(comp).values
y2   = comp['result'].isin(DANGEROUS).astype(int).values

pipe1 = make_pipe(C=0.5, class_weight='balanced')
pipe2 = make_pipe(C=1.0)
pipe1.fit(X1, y1)
pipe2.fit(X2, y2)

# Cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_report(pipe, X, y, label):
    aucs   = cross_val_score(pipe, X, y, cv=skf, scoring='roc_auc')
    briers = -cross_val_score(pipe, X, y, cv=skf, scoring='neg_brier_score')
    print(f'  {label}  n={len(y):>5} | pos={y.sum():>4}')
    print(f'    CV AUC   : {aucs.mean():.3f} ± {aucs.std():.3f}')
    print(f'    CV Brier : {briers.mean():.4f}')

sep = '=' * 55
print(f'{sep}')
print('  EXPECTED CROSS MODEL (xC) — TRAINING')
print(sep)
cv_report(make_pipe(C=0.5, class_weight='balanced'), X1, y1,
          'Stage 1  P(completed)        ')
print()
cv_report(make_pipe(C=1.0), X2, y2,
          'Stage 2  P(dangerous|completed)')
print(sep)

In [ ]:
# ── 8d. Compute xC for every cross ────────────────────────────────────────────
df_xc = df.copy()

# Stage 1
df_xc['xc_stage1'] = pipe1.predict_proba(build_stage1_features(df_xc).values)[:, 1]

# Stage 2: fill missing end coords with column median before scoring
df2 = df_xc.copy()
df2['end_x']    = df2['end_x'].fillna(df2['end_x'].median())
df2['end_y']    = df2['end_y'].fillna(df2['end_y'].median())
df2['angle_deg']= df2['angle_deg'].fillna(0.0)
df_xc['xc_stage2'] = pipe2.predict_proba(build_stage2_features(df2).values)[:, 1]

df_xc['xc'] = df_xc['xc_stage1'] * df_xc['xc_stage2']

print('xC computed.')
print(f'Mean xC overall : {df_xc["xc"].mean():.4f}')
print()
print('Mean xC by result:')
print(df_xc.groupby('result')['xc'].mean().round(4).sort_values(ascending=False).to_string())

In [ ]:
# ── 8e. Diagnostic plots ──────────────────────────────────────────────────────
from matplotlib.colors import LinearSegmentedColormap as LCM

p1_prob = pipe1.predict_proba(X1)[:, 1]
p2_prob = pipe2.predict_proba(X2)[:, 1]
result_order  = ['goal', 'shot_ot', 'shot', 'completed', 'cleared', 'incomplete']
result_colors = [GREEN, '#5BA85A', TEAL, GOLD, '#E07A3A', RED]

fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor(NAVY)
gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)


def ax_style(ax):
    for sp in ['top', 'right']:   ax.spines[sp].set_visible(False)
    for sp in ['left', 'bottom']: ax.spines[sp].set_color('#2A3E52')
    ax.grid(alpha=0.25, color='#2A3E52')


# Calibration
ax1 = fig.add_subplot(gs[0, 0]); ax1.set_facecolor('#111E2B')
for prob, y, lbl, col in [(p1_prob,y1,'Stage 1 P(completed)',TEAL),(p2_prob,y2,'Stage 2 P(danger)',GREEN)]:
    frac, mp = calibration_curve(y, prob, n_bins=8)
    ax1.plot(mp, frac, 'o-', color=col, linewidth=1.8, markersize=4, label=lbl, zorder=3)
ax1.plot([0,1],[0,1],'--',color='#AAAAAA',linewidth=1.0,alpha=0.6,label='Perfect')
ax1.set_xlabel('Predicted probability', fontsize=8); ax1.set_ylabel('Actual fraction', fontsize=8)
ax1.set_title('Calibration', color=GOLD, fontsize=10, fontweight='bold')
ax1.legend(fontsize=7, facecolor='#1A2B3C', edgecolor='#2A3E52'); ax_style(ax1)

# ROC
ax2 = fig.add_subplot(gs[0, 1]); ax2.set_facecolor('#111E2B')
for prob, y, lbl, col in [(p1_prob,y1,'Stage 1',TEAL),(p2_prob,y2,'Stage 2',GREEN)]:
    fpr, tpr, _ = roc_curve(y, prob)
    ax2.plot(fpr, tpr, color=col, linewidth=1.8,
             label=f'{lbl}  AUC={roc_auc_score(y,prob):.3f}', zorder=3)
ax2.plot([0,1],[0,1],'--',color='#AAAAAA',linewidth=1.0,alpha=0.5)
ax2.set_xlabel('FPR', fontsize=8); ax2.set_ylabel('TPR', fontsize=8)
ax2.set_title('ROC Curves', color=GOLD, fontsize=10, fontweight='bold')
ax2.legend(fontsize=7, facecolor='#1A2B3C', edgecolor='#2A3E52'); ax_style(ax2)

# Stage 2 feature coefficients
ax3 = fig.add_subplot(gs[0, 2]); ax3.set_facecolor('#111E2B')
feat_names = ['end_x','end_y','centrality','dist_goal',
              'near_post','far_post','into_box','distance','angle','flight','dest_central']
coefs = pipe2.named_steps['clf'].coef_[0]
order = np.argsort(np.abs(coefs))
ax3.barh(range(len(coefs)), coefs[order],
         color=[GREEN if c > 0 else RED for c in coefs[order]],
         alpha=0.85, edgecolor='#2A3E52', linewidth=0.5)
ax3.set_yticks(range(len(coefs)))
ax3.set_yticklabels([feat_names[i] for i in order], fontsize=7)
ax3.axvline(0, color='#AAAAAA', linewidth=0.8)
ax3.set_title('Stage 2 Coefficients', color=GOLD, fontsize=10, fontweight='bold'); ax_style(ax3)

# xC distribution by result
ax4 = fig.add_subplot(gs[1, 0]); ax4.set_facecolor('#111E2B')
for res, col in zip(result_order, result_colors):
    sub = df_xc[df_xc['result'] == res]['xc']
    if len(sub) > 1:
        ax4.hist(sub, bins=15, alpha=0.5, color=col, label=res, density=True)
ax4.set_xlabel('xC', fontsize=8); ax4.set_ylabel('Density', fontsize=8)
ax4.set_title('xC Distribution by Result', color=GOLD, fontsize=10, fontweight='bold')
ax4.legend(fontsize=6, facecolor='#1A2B3C', edgecolor='#2A3E52'); ax_style(ax4)

# Mean xC by result
ax5 = fig.add_subplot(gs[1, 1]); ax5.set_facecolor('#111E2B')
means  = [df_xc[df_xc['result']==r]['xc'].mean() for r in result_order]
counts = [len(df_xc[df_xc['result']==r]) for r in result_order]
bars   = ax5.bar(result_order, means, color=result_colors, edgecolor='#2A3E52',
                 linewidth=0.6, width=0.65, alpha=0.85)
for bar, cnt in zip(bars, counts):
    ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
             f'n={cnt}', ha='center', va='bottom', fontsize=7, color='#AAAAAA')
ax5.set_ylabel('Mean xC', fontsize=8)
ax5.set_title('Mean xC by Actual Result', color=GOLD, fontsize=10, fontweight='bold')
ax5.tick_params(axis='x', labelsize=7, rotation=20); ax_style(ax5)

# Stage 1 probability surface on pitch
ax6 = fig.add_subplot(gs[1, 2])
pitchx = Pitch(pitch_type='opta', pitch_color=NAVY, line_color='#2A3E52', line_zorder=2)
pitchx.draw(ax=ax6)
gx = np.linspace(40, 100, 60); gy = np.linspace(0, 100, 40)
GX, GY = np.meshgrid(gx, gy)
gf   = np.c_[GX.ravel(), GY.ravel()]
med  = df[['distance_m','flight_metric','curve_score','goal_diff','minute']].median()
gdf  = pd.DataFrame({'x':gf[:,0],'y':gf[:,1],
                      'distance_m':med['distance_m'],'flight_metric':med['flight_metric'],
                      'curve_score':med['curve_score'],'goal_diff':0.0,'minute':45.0})
gc1  = pipe1.predict_proba(build_stage1_features(gdf).values)[:,1].reshape(GX.shape)
cmap_xc = LCM.from_list('xc',[NAVY,'#1B7A78',GOLD,RED])
hm = ax6.pcolormesh(GX, GY, gc1, cmap=cmap_xc, alpha=0.72, shading='gouraud')
plt.colorbar(hm, ax=ax6, shrink=0.7, pad=0.01, label='P(completed)')
pitchx.scatter(df_xc['x'], df_xc['y'], ax=ax6, c=df_xc['xc'],
               cmap=cmap_xc, s=10, alpha=0.55, zorder=4)
ax6.set_title('Stage 1 Surface
P(completed) by origin', color=GOLD, fontsize=9, fontweight='bold')

fig.suptitle('Expected Cross Model (xC) — Eredivisie 2024-25',
             fontsize=14, fontweight='bold', color=WHITE, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 8f. Per-player xC ranking ─────────────────────────────────────────────────
xc_player = (
    df_xc.groupby('player')
    .agg(
        team       = ('team', 'first'),
        crosses    = ('xc', 'count'),
        mean_xc    = ('xc', 'mean'),
        sum_xc     = ('xc', 'sum'),
        mean_s1    = ('xc_stage1', 'mean'),
        mean_s2    = ('xc_stage2', 'mean'),
        vaep_avg   = ('vaep_value', 'mean'),
        rapm       = ('rapm', 'mean'),
        actual_rate= ('result', lambda x: x.isin(DANGEROUS).mean()),
    )
    .reset_index()
    .query(f'crosses >= {MIN_CROSSES}')
    .sort_values('mean_xc', ascending=False)
    .round(4)
)

print(f'Players with ≥{MIN_CROSSES} crosses: {len(xc_player)}')
print()
xc_player.head(20)

In [ ]:
# ── 8g. Top crossers by mean xC — horizontal bar ─────────────────────────────
top_xc = xc_player.nlargest(20, 'mean_xc').sort_values('mean_xc', ascending=True)

fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor(NAVY); ax.set_facecolor('#111E2B')

y = range(len(top_xc))
ax.barh(y, top_xc['mean_xc'], color=TEAL, alpha=0.85,
        edgecolor='#2A3E52', linewidth=0.6, height=0.7)
ax.axvline(df_xc['xc'].mean(), color=GOLD, linestyle='--',
           linewidth=1.2, label=f'League mean xC ({df_xc["xc"].mean():.3f})')

labels = [f"{r['player']} ({r['team']}, n={int(r['crosses'])})"
          for _, r in top_xc.iterrows()]
ax.set_yticks(list(y)); ax.set_yticklabels(labels, fontsize=8, color=WHITE)
ax.set_xlabel('Mean xC per Cross', color='#CCCCCC', fontsize=10)
ax.set_title(f'Top 20 Crossers by xC — Eredivisie 2024-25 (min {MIN_CROSSES} crosses)',
             fontsize=12, fontweight='bold', color=GOLD, pad=12)
ax.legend(fontsize=8, facecolor='#1A2B3C', edgecolor='#2A3E52')
ax.grid(axis='x', alpha=0.25, color='#2A3E52')
for sp in ['top','right']: ax.spines[sp].set_visible(False)
for sp in ['left','bottom']: ax.spines[sp].set_color('#2A3E52')
plt.tight_layout(); plt.show()

In [ ]:
# ── 8h. xC vs actual danger rate scatter (overperformer / underperformer) ─────
fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor(NAVY); ax.set_facecolor('#111E2B')

sc = ax.scatter(
    xc_player['mean_xc'], xc_player['actual_rate'],
    s          = np.clip(xc_player['crosses'] * 4, 30, 300),
    c          = xc_player['vaep_avg'],
    cmap       = 'RdYlGn',
    alpha      = 0.75,
    edgecolors = 'none',
    zorder     = 3,
)
cbar = plt.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label('Avg VAEP', color='#CCCCCC', fontsize=9)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#AAAAAA')

# 45-degree line = actual matches xC exactly
mn = min(xc_player[['mean_xc','actual_rate']].min())
mx = max(xc_player[['mean_xc','actual_rate']].max())
ax.plot([mn, mx], [mn, mx], '--', color='#AAAAAA', linewidth=1.0,
        alpha=0.5, label='xC = actual')
ax.text(mx*0.98, mx*1.02, 'Overperforming ↑', ha='right', fontsize=8,
        color=GREEN, alpha=0.8)
ax.text(mx*0.98, mn*1.1, 'Underperforming ↓', ha='right', fontsize=8,
        color=RED, alpha=0.8)

# Annotate top 10 by crosses
for _, r in xc_player.nlargest(10, 'crosses').iterrows():
    ax.annotate(r['player'], xy=(r['mean_xc'], r['actual_rate']),
                xytext=(5,3), textcoords='offset points',
                fontsize=7, color=WHITE, alpha=0.85)

ax.set_xlabel('Mean xC (expected danger rate)', color='#CCCCCC', fontsize=11)
ax.set_ylabel('Actual danger rate', color='#CCCCCC', fontsize=11)
ax.set_title('xC vs Actual — Over/Underperformers
(bubble = cross volume, colour = VAEP)',
             fontsize=12, fontweight='bold', color=GOLD, pad=12)
ax.legend(fontsize=8, facecolor='#1A2B3C', edgecolor='#2A3E52')
ax.grid(alpha=0.25, color='#2A3E52')
for sp in ['top','right']: ax.spines[sp].set_visible(False)
for sp in ['left','bottom']: ax.spines[sp].set_color('#2A3E52')
plt.tight_layout(); plt.show()

In [ ]:
# ── 8i. Export enriched data with xC back to Drive ────────────────────────────
output_dir = os.path.join('/content/drive/My Drive', 'Event data', 'Eredivisie 2024-2025')
os.makedirs(output_dir, exist_ok=True)

xc_raw_out    = os.path.join(output_dir, 'cross_analysis_xc.csv')
xc_player_out = os.path.join(output_dir, 'cross_xc_by_player.csv')

df_xc.to_csv(xc_raw_out, index=False)
xc_player.to_csv(xc_player_out, index=False)

print(f'xC cross data   → {xc_raw_out}')
print(f'xC player ranks → {xc_player_out}')
print(f'\nNew columns added: xc_stage1, xc_stage2, xc')